In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from pathlib import Path

def clean_numeric(series):
    return (
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '')
        .str.replace(',', '.')
        .astype(float)
    )

def standardize_region_names(df):
    replacements = {
        'Ненецкий авт.округ': 'Ненецкий автономный округ',
        'Hенецкий авт.округ': 'Ненецкий автономный округ',
        '  Ненецкий автономный округ': 'Ненецкий автономный округ',
        'Ямало-Ненецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        'Ямало-Hенецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        '  Ямало-Ненецкий автономный округ': 'Ямало-Ненецкий автономный округ',
        'Ханты-Мансийский авт.округ-Югра': 'Ханты-Мансийский автономный округ - Югра',
        '  Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский автономный округ - Югра',
        'Республика Татарстан(Татарстан)': 'Республика Татарстан',
        'Чувашская Республика(Чувашия)': 'Чувашская Республика',
        'Республика Северная Осетия- Алания': 'Республика Северная Осетия-Алания',
        'Oмская область': 'Омская область',
        'Hижегородская область': 'Нижегородская область',
        'г. Севастополь': 'г.Севастополь',
        'г.Москва': 'г.Москва',
        'г.Санкт-Петербург': 'г.Санкт-Петербург',
        'Чукотский авт.округ': 'Чукотский автономный округ',
    }
    df['Регион'] = df['Регион'].replace(replacements).str.strip()
    return df

def clean_all_numeric_columns(df):
    df = df.copy()
    for col in df.columns:
        if col not in ['Регион', 'Год']:
            df[col] = (
                df[col].astype(str)
                .str.replace('\xa0', '', regex=False)
                .str.replace(' ', '')
                .str.replace(',', '.')
                .str.replace('−', '-')
                .str.replace('–', '-')
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def prepare_features(df, target_col):
    df = df.sort_values(['Регион', 'Год'])
    df[f'lag1_{target_col}'] = df.groupby('Регион')[target_col].shift(1)
    df[f'lag2_{target_col}'] = df.groupby('Регион')[target_col].shift(2)
    df['year_trend'] = df['Год'] - 2014
    return df

df = pd.read_excel("общая_СКР.xlsx")
df = standardize_region_names(df)
df['СКР'] = clean_numeric(df['СКР'])
df['Год'] = pd.to_numeric(df['Год'], errors='coerce').astype(int)
df = df[(df['Год'] >= 2014) & (df['Год'] <= 2023)]
df = clean_all_numeric_columns(df)

df_prep = prepare_features(df, 'СКР')
train = df_prep[df_prep['Год'] <= 2022]
test = df_prep[df_prep['Год'] == 2023]

feature_cols = [col for col in df_prep.columns if col not in ['Регион', 'Год', 'СКР']]
feature_cols = [f for f in feature_cols if f in train.columns]

X_train = train[feature_cols].fillna(0)
y_train = train['СКР']
X_test = test[feature_cols].fillna(0)
y_test = test['СКР']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

results = []
for i, (_, row) in enumerate(test.iterrows()):
    actual = y_test.iloc[i]
    pred = y_pred[i]
    abs_err = abs(pred - actual)
    results.append({
        'Регион': row['Регион'],
        'Прогноз_2023': round(pred, 4),
        'Факт_2023': round(actual, 4),
        'Абсолютная_ошибка': round(abs_err, 4),
        'Относительная_ошибка_%': round(abs_err / actual * 100, 2),
        'RMSE': round(abs_err, 4),
        'MAE': round(abs_err, 4)
    })

df_results = pd.DataFrame(results)
mean_abs = df_results['Абсолютная_ошибка'].mean()
mean_rel = df_results['Относительная_ошибка_%'].mean()
global_rmse = np.sqrt(mean_squared_error(df_results['Факт_2023'], df_results['Прогноз_2023']))
global_mae = mean_absolute_error(df_results['Факт_2023'], df_results['Прогноз_2023'])

summary = pd.DataFrame([{
    'Регион': '=== ИТОГО ===',
    'Прогноз_2023': '',
    'Факт_2023': '',
    'Абсолютная_ошибка': round(mean_abs, 4),
    'Относительная_ошибка_%': round(mean_rel, 2),
    'RMSE': round(global_rmse, 4),
    'MAE': round(global_mae, 4)
}])

final_df = pd.concat([df_results, summary], ignore_index=True)
final_df.to_excel("xgboost_СКР_валидация_2023.xlsx", index=False)
print("Файл сохранён: xgboost_СКР_валидация_2023.xlsx")